# Qiskit Fall Fest 2026 — Phase 1, PS-O2: Portfolio Optimization

This notebook uses the repository's shared `src/` modules to formulate the four-asset portfolio problem as a QUBO, verify the encoding against exhaustive classical enumeration, evaluate a depth-one QAOA circuit, and compare the reference Processor A and Processor B coupling maps using the prescribed transpilation settings and synthetic noise model.

**Reproducibility:** run all cells from top to bottom in a fresh Google Colab runtime. The notebook downloads the public GitHub repository archive automatically, imports the shared modules, and saves comparison tables, metadata, and figures under `results/`.

**Parameter transparency:** the problem guide specifies the problem inputs and budget constraint, but the experiment's numerical risk-aversion `q` and penalty `λ` must be verified against the guide and are currently configured in `src/problem.py`. Results are conditional on these settings and the stated depth-one QAOA ansatz.


## 1. Environment and imports

The notebook installs its Python dependencies, downloads and extracts the public GitHub repository archive in Colab, and imports the shared implementation from `src/`. Run all cells from top to bottom in a fresh runtime.


In [ ]:
%pip install -q "qiskit==2.5.2" "qiskit-aer==0.17.2" scipy pandas matplotlib

In [ ]:
import sys
import os
import shutil
import zipfile
import urllib.request
from pathlib import Path

REPO_ZIP_URL = (
    "https://codeload.github.com/"
    "sandhya313-hub/qiskit-fallfest-ps-o2/zip/refs/heads/main"
)

ZIP_PATH = Path("/content/qiskit-project.zip")
EXTRACT_DIR = Path("/content/qiskit-project")

print("Downloading project repository from GitHub...")
urllib.request.urlretrieve(REPO_ZIP_URL, ZIP_PATH)

if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError(
        "GitHub did not return a valid ZIP. Check that the repository is public "
        "and that the branch name in REPO_ZIP_URL is correct."
    )

# Remove any previous extraction so stale files cannot be selected.
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

matches = list(EXTRACT_DIR.rglob("src/problem.py"))
if not matches:
    raise FileNotFoundError(
        "src/problem.py was not found in the downloaded repository."
    )

PROJECT_ROOT = matches[0].parent.parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_files = [
    PROJECT_ROOT / "processors" / "processor_A.json",
    PROJECT_ROOT / "processors" / "processor_B.json",
]

missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Required project files are missing:\n" + "\n".join(missing_files)
    )

print("Repository downloaded and extracted successfully!")
print("Project root:", PROJECT_ROOT)
print("Problem module:", PROJECT_ROOT / "src" / "problem.py")
print("Processor configurations verified.")


In [ ]:

import sys
from pathlib import Path

# Reuse the repository path from the successful setup cell.
PROJECT_ROOT = Path(PROJECT_ROOT)

if not (PROJECT_ROOT / "src" / "problem.py").is_file():
    raise FileNotFoundError(
        f"Cannot find src/problem.py at {PROJECT_ROOT}. "
        "Run the GitHub repository download setup cell first."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.problem import (
    NUM_ASSETS,
    RETURNS,
    SIGMA,
    RISK_AVERSION,
    BUDGET_PENALTY,
    TARGET_ASSETS,
    QAOA_P,
    load_problem,
    portfolio_objective,
    qubo_coefficients,
    qubo_energy as module_qubo_energy,
)

from src.processors import (
    load_processor_config,
    get_coupling_edges,
    get_processor_summary,
)

from src.routing import (
    BASIS_GATES,
    TRANSPILE_SEED,
    OPTIMIZATION_LEVEL,
    compare_processor_transpilation,
)

from src.metrics import compare_circuit_metrics
from src.visualization import (
    plot_processor_comparison,
    plot_metric_comparison,
)

print("All project modules imported successfully!")
print("Repository:", PROJECT_ROOT)


## 2. Problem inputs

The notebook loads returns, the risk matrix, and the cardinality target from `src/problem.py`, the single source of truth for the experiment. Verify that these numerical inputs match the official problem guide before submission; values in the module are the configured benchmark inputs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)
print("Imports successful!")

In [ ]:
# Canonical problem inputs come from src/problem.py.
problem = load_problem()
RETURNS = problem.returns
SIGMA = problem.sigma
BUDGET = problem.target_assets

assert RETURNS.shape == (NUM_ASSETS,)
assert SIGMA.shape == (NUM_ASSETS, NUM_ASSETS)
assert np.allclose(SIGMA, SIGMA.T)
assert np.all(np.isfinite(RETURNS)) and np.all(np.isfinite(SIGMA))
assert BUDGET == 2

print("Expected returns:", RETURNS.tolist())
print("Budget (assets selected):", BUDGET)
print("Risk matrix source:", problem.matrix_source)
print("Covariance matrix:\\n", SIGMA)
print("Covariance eigenvalues:", np.linalg.eigvalsh(SIGMA))


## 3. Objective and feasible portfolios

We use the guide’s objective form `wᵀΣw − q μᵀw` subject to `sum(wᵢ) = k`. Since the numerical risk-aversion value and penalty are not specified in the guide, the values below are experimental choices. The penalty converts the equality constraint into an unconstrained binary objective.

In [ ]:
import itertools
import numpy as np
import pandas as pd

# These parameter values are centralized in src/problem.py
RISK_AVERSION = problem.risk_aversion
PENALTY = problem.budget_penalty
QAOA_P = 1

def direct_objective(w):
    """Evaluate the canonical objective implemented in src/problem.py."""
    return portfolio_objective(
        w, problem=problem, include_budget_penalty=True
    )

all_bits = list(itertools.product([0, 1], repeat=NUM_ASSETS))

## 4. QUBO encoding and verification

We use the upper-triangular convention `E(x) = offset + Σᵢ Qᵢᵢxᵢ + Σᵢ<ⱼ Qᵢⱼxᵢxⱼ`. The offset is retained for reporting, although a constant offset does not affect QAOA measurement probabilities.

In [ ]:
# Obtain the canonical upper-triangular QUBO and offset from src/problem.py.
Q, Q_OFFSET = qubo_coefficients(problem)


def qubo_energy(bits):
    """Evaluate the canonical QUBO energy using the module implementation."""
    return module_qubo_energy(bits, problem=problem)


qubo_errors = [abs(qubo_energy(bits) - direct_objective(bits)) for bits in all_bits]
print("QUBO matrix:\\n", Q)
print("Constant offset:", Q_OFFSET)
print("Maximum absolute difference over all 16 bitstrings:", max(qubo_errors))
assert max(qubo_errors) < 1e-9
print("PASS: QUBO energy matches the direct objective for all 16 bitstrings.")


## 5. Exact classical baseline

All 16 binary strings are enumerated. The exact solution is used as the reference for evaluating QAOA; it is not a quantum result.

In [ ]:
classical_results = []
for bits in all_bits:
    x = np.asarray(bits, dtype=float)
    feasible = int(x.sum()) == BUDGET
    classical_results.append({
        "bits": list(map(int, bits)),
        "assets_zero_indexed": [i for i, bit in enumerate(bits) if bit],
        "selected_count": int(x.sum()),
        "return": float(RETURNS @ x),
        "risk": float(x @ SIGMA @ x),
        "objective": direct_objective(bits),
        "qubo_energy": qubo_energy(bits),
        "feasible": feasible,
    })

classical_df = pd.DataFrame(classical_results)
classical_optimum = min(classical_results, key=lambda row: row["objective"])
best_feasible = min((row for row in classical_results if row["feasible"]),
                    key=lambda row: row["objective"])
display(classical_df.sort_values("objective").head(8))
print("Exact penalized optimum:", classical_optimum)
print("Best feasible portfolio:", best_feasible)
assert classical_optimum["feasible"]
assert classical_optimum["bits"] == best_feasible["bits"]
print("PASS: the exact penalized optimum is feasible and agrees with the best feasible portfolio.")

## 6. Build the depth-one QAOA circuit

The cost unitary is derived from the upper-triangular QUBO convention above. This notebook uses a standard transverse-field mixer and one QAOA layer (`p = 1`).

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

print("QAOA circuit imports successful!")

In [ ]:
def build_qaoa_circuit(Q_matrix):
    Q_matrix = np.asarray(Q_matrix, dtype=float)
    if Q_matrix.shape != (NUM_ASSETS, NUM_ASSETS):
        raise ValueError("Q matrix must be 4x4.")
    if not np.all(np.isfinite(Q_matrix)):
        raise ValueError("Q matrix must contain finite values.")

    gamma = Parameter("gamma")
    beta = Parameter("beta")
    qc = QuantumCircuit(NUM_ASSETS)
    qc.h(range(NUM_ASSETS))

    # Convert x_i = (1 - Z_i)/2 into a cost Hamiltonian.
    z_coeff = np.zeros(NUM_ASSETS, dtype=float)
    for i in range(NUM_ASSETS):
        z_coeff[i] = -Q_matrix[i, i] / 2
    for i in range(NUM_ASSETS):
        for j in range(i + 1, NUM_ASSETS):
            coeff = Q_matrix[i, j]
            z_coeff[i] -= coeff / 4
            z_coeff[j] -= coeff / 4
            if not np.isclose(coeff, 0.0):
                qc.cx(i, j)
                qc.rz(gamma * coeff / 2, j)
                qc.cx(i, j)
    for i in range(NUM_ASSETS):
        if not np.isclose(z_coeff[i], 0.0):
            qc.rz(2 * gamma * z_coeff[i], i)
    for i in range(NUM_ASSETS):
        qc.rx(2 * beta, i)
    qc.measure_all()
    return qc

circuit = build_qaoa_circuit(Q)
print("Logical qubits:", circuit.num_qubits)
print("QAOA depth p:", QAOA_P)
print("Parameters:", sorted(p.name for p in circuit.parameters))
print("Logical circuit depth:", circuit.depth())
print("Logical operations:", dict(circuit.count_ops()))

## 7. Ideal statevector optimization

This parameter search is an ideal, noiseless simulation. It is not a Processor A/B result. Differential Evolution is seeded for reproducibility; a successful optimizer termination does not guarantee a global optimum.

In [ ]:
import numpy as np
from scipy.optimize import differential_evolution
from qiskit.quantum_info import Statevector

print("Optimization imports successful!")

In [ ]:
SEED = 42

print("Random seed configured:", SEED)

In [ ]:
def expected_qubo_cost(params):
    gamma_value, beta_value = params
    parameter_map = {p.name: p for p in circuit.parameters}
    bound = circuit.assign_parameters({
        parameter_map["gamma"]: float(gamma_value),
        parameter_map["beta"]: float(beta_value),
    })
    unitary = bound.remove_final_measurements(inplace=False)
    probs = Statevector.from_instruction(unitary).probabilities()
    return float(sum(
        prob * qubo_energy([(index >> i) & 1 for i in range(NUM_ASSETS)])
        for index, prob in enumerate(probs)
    ))

optimizer_result = differential_evolution(
    expected_qubo_cost,
    bounds=[(0.0, 0.2), (0.0, np.pi)],
    seed=SEED,
    maxiter=100,
    popsize=12,
    tol=1e-7,
    polish=True,
    workers=1,
)
GAMMA_BEST = float(optimizer_result.x[0])
BETA_BEST = float(optimizer_result.x[1])
IDEAL_EXPECTED_ENERGY = float(optimizer_result.fun)
print("Best gamma:", GAMMA_BEST)
print("Best beta:", BETA_BEST)
print("Ideal expected energy:", IDEAL_EXPECTED_ENERGY)
print("Optimizer success:", bool(optimizer_result.success))
print("Optimizer message:", optimizer_result.message)
print("Exact classical optimum energy:", classical_optimum["objective"])
print("Absolute energy gap:", abs(IDEAL_EXPECTED_ENERGY - classical_optimum["objective"]))

## 8. Ideal output distribution and baseline quality

These probabilities come from the ideal statevector, not from finite-shot noisy simulation.

In [ ]:
from pathlib import Path

TABLES_DIR = PROJECT_ROOT / "results" / "tables"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Tables directory:", TABLES_DIR)
print("Figures directory:", FIGURES_DIR)

In [ ]:
parameter_map = {p.name: p for p in circuit.parameters}
bound = circuit.assign_parameters({
    parameter_map["gamma"]: GAMMA_BEST,
    parameter_map["beta"]: BETA_BEST,
})
unitary = bound.remove_final_measurements(inplace=False)
ideal_probabilities = Statevector.from_instruction(unitary).probabilities()

ideal_rows = []
for index, probability in enumerate(ideal_probabilities):
    bits = [(index >> i) & 1 for i in range(NUM_ASSETS)]
    x = np.asarray(bits, dtype=float)
    ideal_rows.append({
        "bitstring": "".join(map(str, bits)),
        "bits": bits,
        "probability": float(probability),
        "selected_count": int(x.sum()),
        "return": float(RETURNS @ x),
        "risk": float(x @ SIGMA @ x),
        "energy": qubo_energy(bits),
        "feasible": int(x.sum()) == BUDGET,
    })
ideal_df = pd.DataFrame(ideal_rows).sort_values("probability", ascending=False)
display(ideal_df.head(8))

optimal_bits = classical_optimum["bits"]
optimal_probability = float(ideal_probabilities[sum(bit << i for i, bit in enumerate(optimal_bits))])
feasible_probability = float(ideal_df.loc[ideal_df["feasible"], "probability"].sum())
print(f"Ideal probability of exact classical optimum: {optimal_probability:.4%}")
print(f"Ideal probability of any feasible portfolio: {feasible_probability:.4%}")
print(f"Ideal probability of infeasible portfolios: {1-feasible_probability:.4%}")
ideal_df.to_csv(TABLES_DIR / "ideal_qaoa_distribution.csv", index=False)

## 9. Processor A/B reference topologies and transpilation

Processor topologies are loaded from `processors/processor_A.json` and `processors/processor_B.json` through `src/processors.py`. The notebook calls `src/routing.py` to transpile the same logical circuit with the shared basis gates, optimization level, and transpiler seed, then calls `src/metrics.py` for comparable circuit metrics. The topologies are reference comparison models, not claims about real hardware.


In [ ]:
# Load topologies from processors/processor_A.json and processor_B.json.
processor_a_config = load_processor_config("A")
processor_b_config = load_processor_config("B")
PROCESSOR_A_EDGES = get_coupling_edges("A")
PROCESSOR_B_EDGES = get_coupling_edges("B")

assert processor_a_config["qubit_count"] == 5
assert processor_b_config["qubit_count"] == 9
assert len(PROCESSOR_A_EDGES) == 4
assert len(PROCESSOR_B_EDGES) == 12

transpiled_circuits = compare_processor_transpilation(circuit)
transpiled_reference = transpiled_circuits["reference"]
transpiled_A = transpiled_circuits["processor_A"]
transpiled_B = transpiled_circuits["processor_B"]

metric_results = compare_circuit_metrics(
    transpiled_reference, transpiled_A, transpiled_B
)

def transpilation_metrics(name, circuit_metrics):
    return {
        "Processor": name,
        "Physical qubits in circuit": circuit_metrics["physical_qubits"],
        "Depth": circuit_metrics["circuit_depth"],
        "Reference CX": circuit_metrics["reference_cx"],
        "Transpiled CX": circuit_metrics["transpiled_cx"],
        "Additional CX vs reference": circuit_metrics["additional_cx"],
        "Explicit SWAP instructions after basis translation": circuit_metrics["explicit_swap_count"],
        "SWAP-equivalent estimate (additional CX / 3)": circuit_metrics["swap_equivalent_estimate"],
        "RZ": circuit_metrics["rz_count"],
        "SX": circuit_metrics["sx_count"],
        "X": circuit_metrics["x_count"],
    }

transpilation_df = pd.DataFrame([
    transpilation_metrics("Unconstrained reference", metric_results["reference"]),
    transpilation_metrics("Processor A", metric_results["processor_A"]),
    transpilation_metrics("Processor B", metric_results["processor_B"]),
])
display(transpilation_df)
transpilation_df.to_csv(TABLES_DIR / "transpilation_metrics.csv", index=False)

print("Processor A configuration:", get_processor_summary("A"))
print("Processor B configuration:", get_processor_summary("B"))
print("Processor A edges:", PROCESSOR_A_EDGES)
print("Processor B edges:", PROCESSOR_B_EDGES)
print("Note: explicit SWAP instructions may be zero because SWAPs are decomposed into basis gates.")
print("The SWAP-equivalent value is additional CX / 3, a rough estimate rather than an exact routing SWAP count.")

# Use the shared visualization module for topology diagrams.
fig, _ = plot_processor_comparison()
fig.savefig(FIGURES_DIR / "processor_topologies.png", dpi=160, bbox_inches="tight")
plt.show()


## 10. Required synthetic noise model

The synthetic model applies single-qubit depolarizing error `p = 0.001` to each one-qubit basis gate (`rz`, `sx`, and `x`), two-qubit depolarizing error `p = 0.005` to `cx`, and symmetric readout error `p = 0.01`. This is a simulator model, not an execution on physical hardware.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    ReadoutError,
)

print("Noise simulation imports successful!")

In [ ]:
SHOTS = 4096

print("Shots per processor:", SHOTS)

In [ ]:
P_1Q = 0.001
P_CX = 0.005
P_READOUT = 0.01

noise_model = NoiseModel()
error_1q = depolarizing_error(P_1Q, 1)
error_cx = depolarizing_error(P_CX, 2)
readout_error = ReadoutError([
    [1 - P_READOUT, P_READOUT],
    [P_READOUT, 1 - P_READOUT],
])
noise_model.add_all_qubit_quantum_error(error_1q, ["rz", "sx", "x"])
noise_model.add_all_qubit_quantum_error(error_cx, ["cx"])
noise_model.add_all_qubit_readout_error(readout_error)

simulator = AerSimulator(noise_model=noise_model, basis_gates=BASIS_GATES)
print("Noise model basis gates:", noise_model.basis_gates)
print("1-qubit depolarizing probability:", P_1Q)
print("CX depolarizing probability:", P_CX)
print("Readout error probability:", P_READOUT)
print("Shots per processor:", SHOTS)

## 11. Noisy 4096-shot Processor A/B benchmark

The optimized ideal parameters are held fixed for both topologies so the comparison changes the topology and resulting transpilation/noise exposure—not the optimizer settings. Accuracy is reported as (1) exact-optimum hit rate and (2) feasible-output rate among sampled bitstrings. The sampled best feasible objective and empirical mean objective are also reported.

In [ ]:
def bind_best_parameters(transpiled_circuit):
    pmap = {p.name: p for p in transpiled_circuit.parameters}
    return transpiled_circuit.assign_parameters({
        pmap["gamma"]: GAMMA_BEST,
        pmap["beta"]: BETA_BEST,
    })


def parse_count_bitstring(count_key, num_assets=NUM_ASSETS):
    # Qiskit count strings display the highest-index classical bit on the left.
    compact = count_key.replace(" ", "")
    if len(compact) < num_assets:
        compact = compact.zfill(num_assets)
    return [int(compact[-1 - i]) for i in range(num_assets)]


def run_noisy_benchmark(name, transpiled_circuit):
    bound_physical = bind_best_parameters(transpiled_circuit)
    result = simulator.run(bound_physical, shots=SHOTS, seed_simulator=SEED).result()
    counts = result.get_counts()
    total = sum(counts.values())
    assert total == SHOTS, f"Expected {SHOTS} shots but got {total}."

    optimum_key = tuple(classical_optimum["bits"])
    optimal_count = 0
    feasible_count = 0
    weighted_energy = 0.0
    best_sampled_feasible = None
    distribution = []

    for key, count in counts.items():
        bits = parse_count_bitstring(key)
        x = np.asarray(bits, dtype=float)
        is_feasible = int(x.sum()) == BUDGET
        energy = qubo_energy(bits)
        probability = count / total
        weighted_energy += probability * energy
        if tuple(bits) == optimum_key:
            optimal_count += count
        if is_feasible:
            feasible_count += count
            row = {
                "bits": bits,
                "count": int(count),
                "probability": probability,
                "return": float(RETURNS @ x),
                "risk": float(x @ SIGMA @ x),
                "objective": direct_objective(bits),
                "energy": energy,
            }
            if best_sampled_feasible is None or row["objective"] < best_sampled_feasible["objective"]:
                best_sampled_feasible = row
        distribution.append({
            "processor": name,
            "count_key": key,
            "bits": bits,
            "count": int(count),
            "probability": probability,
            "selected_count": int(x.sum()),
            "feasible": is_feasible,
            "energy": energy,
        })

    summary = {
        "Processor": name,
        "Shots": total,
        "Optimal portfolio hits": optimal_count,
        "Optimal portfolio hit rate": optimal_count / total,
        "Feasible samples": feasible_count,
        "Feasible-output rate": feasible_count / total,
        "Infeasible-output rate": 1 - feasible_count / total,
        "Empirical mean objective": weighted_energy,
        "Best sampled feasible bits": (best_sampled_feasible["bits"] if best_sampled_feasible else None),
        "Best sampled feasible objective": (best_sampled_feasible["objective"] if best_sampled_feasible else None),
    }
    return summary, distribution

summary_A, distribution_A = run_noisy_benchmark("Processor A", transpiled_A)
summary_B, distribution_B = run_noisy_benchmark("Processor B", transpiled_B)
noisy_summary_df = pd.DataFrame([summary_A, summary_B])
distribution_df = pd.DataFrame(distribution_A + distribution_B)
display(noisy_summary_df)
noisy_summary_df.to_csv(TABLES_DIR / "noisy_processor_comparison.csv", index=False)
distribution_df.to_csv(TABLES_DIR / "noisy_shot_distribution.csv", index=False)

assert noisy_summary_df["Shots"].eq(SHOTS).all()
print("PASS: both processor simulations completed with 4096 shots each.")

## 12. Combined comparison table and figures

The table combines transpilation metrics and noisy sampling quality. Because this is a stochastic simulation, small differences can occur when the seed, shot count, or noise assumptions change.

In [ ]:
processor_metrics_df = transpilation_df[transpilation_df["Processor"].isin(["Processor A", "Processor B"])].copy()
combined_df = processor_metrics_df.merge(noisy_summary_df, on="Processor", how="inner")
display(combined_df)
combined_df.to_csv(TABLES_DIR / "processor_ab_final_comparison.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(combined_df["Processor"], combined_df["Depth"])
axes[0].set_title("Transpiled circuit depth")
axes[0].set_ylabel("Depth")
axes[1].bar(combined_df["Processor"], combined_df["Transpiled CX"])
axes[1].set_title("Transpiled CX gate count")
axes[1].set_ylabel("CX gates")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "processor_ab_circuit_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(combined_df["Processor"], combined_df["Optimal portfolio hit rate"] * 100)
axes[0].set_title("Exact-optimum hit rate")
axes[0].set_ylabel("Percent of shots (%)")
axes[1].bar(combined_df["Processor"], combined_df["Feasible-output rate"] * 100)
axes[1].set_title("Feasible-output rate")
axes[1].set_ylabel("Percent of shots (%)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "processor_ab_noisy_quality.png", dpi=160, bbox_inches="tight")
plt.show()

print("Saved tables to:", TABLES_DIR.resolve())
print("Saved figures to:", FIGURES_DIR.resolve())

## 13. Export reproducibility metadata and final checks

This cell writes the experimental assumptions, noise rates, optimizer parameters, and summary results to JSON for the report. Do not replace actual printed metrics with estimates in the final report.

In [ ]:
from pathlib import Path

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Results directory:", RESULTS_DIR)

In [ ]:
import json
metadata = {
    "problem_statement": "Qiskit Fall Fest 2026 Phase 1 PS-O2: Portfolio Optimization",
    "official_inputs": {
        "returns": RETURNS.tolist(),
        "budget": BUDGET,
        "covariance_matrix": SIGMA.tolist(),
    },
    "experimental_assumptions": {
        "risk_aversion_q": RISK_AVERSION,
        "qubo_penalty_lambda": PENALTY,
        "qaoa_depth_p": QAOA_P,
        "parameter_bounds": {"gamma": [0.0, 0.2], "beta": [0.0, float(np.pi)]},
        "optimizer": "SciPy differential_evolution",
        "optimizer_seed": SEED,
    },
    "transpilation": {
        "basis_gates": BASIS_GATES,
        "optimization_level": OPTIMIZATION_LEVEL,
        "seed_transpiler": TRANSPILE_SEED,
    },
    "noise_model": {
        "one_qubit_depolarizing_probability": P_1Q,
        "cx_depolarizing_probability": P_CX,
        "readout_error_probability": P_READOUT,
        "shots_per_processor": SHOTS,
    },
    "classical_optimum": classical_optimum,
    "ideal_qaoa": {
        "gamma": GAMMA_BEST,
        "beta": BETA_BEST,
        "expected_energy": IDEAL_EXPECTED_ENERGY,
        "optimal_bitstring_probability": optimal_probability,
        "feasible_output_probability": feasible_probability,
    },
    "noisy_processor_summary": noisy_summary_df.to_dict(orient="records"),
    "transpilation_summary": transpilation_df.to_dict(orient="records"),
}
with open(TABLES_DIR / "benchmark_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

assert max(qubo_errors) < 1e-9
assert classical_optimum["feasible"]
assert set(noisy_summary_df["Processor"]) == {"Processor A", "Processor B"}
assert noisy_summary_df["Shots"].eq(SHOTS).all()
assert np.isfinite(noisy_summary_df["Optimal portfolio hit rate"]).all()

print("ALL NOTEBOOK CHECKS PASSED")
print("Files created:")
for path in sorted(RESULTS_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.as_posix())

## 14. Interpretation and limitations

- The exact classical enumeration is the reference optimum for the chosen `q = 2.0` and `λ = 20.0`; those parameter values are experimental assumptions, not numerical values prescribed by the problem guide.
- The QAOA ansatz is depth one. Its probability of sampling the optimum may be modest; report the measured values rather than claiming quantum advantage.
- Processor A and B are simulated reference coupling maps, not physical QPUs. The noisy benchmark is a synthetic simulation using the rates documented above.
- `Explicit SWAP instructions after basis translation` may be zero because SWAPs are decomposed into the allowed basis. `SWAP-equivalent estimate` is additional CX count divided by three and is only a rough overhead indicator, not an exact routing-stage SWAP count.
- The results are specific to the optimizer bounds, seed, noise model, transpiler settings, and finite shot count. Re-run all cells to regenerate them.
- Before final submission, ensure `README.md`, `report.md`, `requirements.txt`, the processor JSON files, and the saved results agree with the values generated by this notebook.

In [ ]:
from pathlib import Path

results_dir = PROJECT_ROOT / "results"

print("Results folder:", results_dir)

for path in sorted(results_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))

In [ ]:
import shutil
from google.colab import files

results_dir = PROJECT_ROOT / "results"
zip_path = shutil.make_archive(
    "/content/qiskit-results",
    "zip",
    root_dir=PROJECT_ROOT,
    base_dir="results"
)

files.download(zip_path)